
# LIBROS Y BYTES

Eres contratado/a por una pequeña cadena de librerías llamada **"Libros & Bytes"** para desarrollar un
sistema que gestione su inventario y permita a los usuarios simular una compra en línea. Trabajarás
solo en la lógica del sistema sin preocuparte de la interfaz visual. El sistema debe cumplir con los
siguientes requerimientos y funcionalidades.

---

## Requerimientos

### 1. Definir variables básicas y tipos de datos (1 punto)
- Crea una **lista** que contenga al menos **cinco libros**, donde cada libro sea un **diccionario** con los atributos:
  - `título` (cadena de caracteres)
  - `autor` (cadena de caracteres)
  - `precio` (decimal)
  - `stock` (entero)

### 2. Control de flujo (1 punto)
- Implementa una función llamada `mostrar_libros_disponibles()` que recorra la lista de
libros y **muestre** los libros que tienen **más de una unidad en stock** usando un `for` y una condición `if`.

### 3. Condiciones y operadores (1 punto)
- Solicita al usuario que ingrese un **rango de precios (mínimo y máximo)** y utiliza una
sentencia `if / elif / else` para **filtrar** los libros en el rango ingresado y mostrarlos.

### 4. Función personalizada para simular una compra (2 puntos)
- Crea una función `comprar_libros(título, cantidad)` que:
  - Verifique si el libro está en el inventario y si la cantidad deseada está disponible.
  - Si la compra es válida, **reste** la cantidad comprada al stock y muestre el **monto total** de la compra.
  - Si la cantidad solicitada es mayor al stock disponible, muestre un **mensaje de error**.

### 5. Uso de bucle `while` hasta que el usuario decida salir (1 punto)
- Implementa un bucle `while` que permita realizar **múltiples compras** hasta que el usuario ingrese una opción de salida.

### 6. Estructura de datos, gestión de **descuentos** (2 puntos)
- Usa un **diccionario** para almacenar **descuentos especiales por autor** (por ejemplo, 10% para un autor).
- En `comprar_libros`, verifica si el autor tiene descuento y **aplíc**alo al monto total si corresponde.

### 7. Simulación de una **factura** (2 puntos)
- Al finalizar la compra, muestra un **resumen** con el total de **libros comprados**, el **monto total pagado** y el **ahorro por descuentos**.

> **Tip:** Este notebook usa celdas Markdown (como esta) para explicaciones y celdas de **Código** para la lógica en Python.


In [ ]:

# =============================================
# 1) Inventario base: lista de diccionarios
# =============================================
from typing import List, Dict, Tuple

inventario: List[Dict] = [
    {"título": "El nombre del viento", "autor": "Patrick Rothfuss", "precio": 15990.0, "stock": 5},
    {"título": "Cien años de soledad", "autor": "Gabriel García Márquez", "precio": 12990.0, "stock": 2},
    {"título": "La sombra del viento", "autor": "Carlos Ruiz Zafón", "precio": 11990.0, "stock": 0},
    {"título": "Ficciones", "autor": "Jorge Luis Borges", "precio": 8990.0, "stock": 3},
    {"título": "Rayuela", "autor": "Julio Cortázar", "precio": 10990.0, "stock": 1},
    # Puedes agregar más libros si lo deseas
]

# Descuentos por autor (ejemplo)
descuentos_por_autor = {
    "Gabriel García Márquez": 0.10,  # 10%
    "Jorge Luis Borges": 0.05,       # 5%
    # Agrega más autores y descuentos si quieres
}

# Carrito y acumuladores para la factura final
carrito: List[Tuple[str, int, float, float]] = []  # (título, cantidad, subtotal_con_desc, ahorro)
total_libros_comprados = 0
total_pagado = 0.0
total_ahorro = 0.0


In [ ]:

# =============================================
# 2) Mostrar libros disponibles (> 1 en stock)
# =============================================
def mostrar_libros_disponibles():
    print("📚 Libros con más de 1 unidad en stock:")
    disponibles = False
    for libro in inventario:
        if libro["stock"] > 1:
            disponibles = True
            print(f"- {libro['título']} | Autor: {libro['autor']} | Precio: ${libro['precio']:.0f} | Stock: {libro['stock']}")
    if not disponibles:
        print("No hay libros con más de 1 unidad en stock.")
        
# Ejemplo de uso (puedes ejecutar esta celda para ver la lista):
# mostrar_libros_disponibles()


In [ ]:

# =============================================
# 3) Filtrar por rango de precios usando if/elif/else
# =============================================
def filtrar_por_rango_precios():
    try:
        minimo = float(input("Ingresa precio mínimo: ").strip())
        maximo = float(input("Ingresa precio máximo: ").strip())
    except ValueError:
        print("⚠️ Entrada inválida. Debes ingresar números (ej: 10000).")
        return
    
    if minimo < 0 and maximo < 0:
        print("⚠️ Ambos valores son negativos. No se puede filtrar.")
    elif minimo > maximo:
        print("⚠️ El mínimo es mayor que el máximo. Intercambiando valores...")
        minimo, maximo = maximo, minimo
        # sigue al else para filtrar correctamente
        # (no hacemos 'return' porque aún podemos mostrar resultados)
        # cae al else automáticamente
        print(f"Nuevo rango: {minimo} a {maximo}")
        # intencionalmente sin 'return' para continuar
        count = 0
        for libro in inventario:
            if minimo <= libro["precio"] <= maximo:
                print(f"- {libro['título']} | ${libro['precio']:.0f} | Stock: {libro['stock']}")
                count += 1
        if count == 0:
            print("No se encontraron libros en ese rango de precios.")
    else:
        # Caso normal
        count = 0
        for libro in inventario:
            if minimo <= libro["precio"] <= maximo:
                print(f"- {libro['título']} | ${libro['precio']:.0f} | Stock: {libro['stock']}")
                count += 1
        if count == 0:
            print("No se encontraron libros en ese rango de precios.")
            
# Ejemplo de uso interactivo:
# filtrar_por_rango_precios()


In [ ]:

# =============================================
# 4) Comprar libros con gestión de stock y descuentos
# =============================================
def comprar_libros(título: str, cantidad: int):
    global total_libros_comprados, total_pagado, total_ahorro
    
    # Buscar el libro en el inventario
    libro = next((l for l in inventario if l["título"].lower() == título.lower()), None)
    if libro is None:
        print(f"❌ El libro '{título}' no está en el inventario.")
        return
    
    # Verificar stock
    if cantidad <= 0:
        print("⚠️ La cantidad debe ser mayor que 0.")
        return
        
    if cantidad > libro["stock"]:
        print(f"❌ Stock insuficiente. Disponible: {libro['stock']} unidades.")
        return
    
    # Calcular subtotal y descuento por autor (si aplica)
    precio_unitario = libro["precio"]
    subtotal = precio_unitario * cantidad
    
    descuento = descuentos_por_autor.get(libro["autor"], 0.0)
    ahorro = subtotal * descuento
    total_con_descuento = subtotal - ahorro
    
    # Actualizar stock
    libro["stock"] -= cantidad
    
    # Actualizar acumuladores y carrito
    total_libros_comprados += cantidad
    total_pagado += total_con_descuento
    total_ahorro += ahorro
    carrito.append((libro["título"], cantidad, total_con_descuento, ahorro))
    
    # Mostrar detalle de la compra
    if descuento > 0:
        print(f"✅ Compra realizada: '{libro['título']}' x{cantidad}")
        print(f"   Subtotal: ${subtotal:,.0f} | Descuento autor {descuento*100:.0f}%: -${ahorro:,.0f}")
        print(f"   Total a pagar por este ítem: ${total_con_descuento:,.0f}")
    else:
        print(f"✅ Compra realizada: '{libro['título']}' x{cantidad}")
        print(f"   Total a pagar por este ítem (sin descuento): ${total_con_descuento:,.0f}")


In [ ]:

# =============================================
# 5) Bucle while para múltiples compras
# 7) Factura al finalizar
# =============================================
def ejecutar_carrito():
    print("Bienvenido/a a Libros & Bytes 🧾")
    print("Comandos: 'ver' (disponibles), 'rango' (filtrar por precio), 'comprar', 'salir'")
    while True:
        cmd = input("\n¿Qué deseas hacer? (ver/rango/comprar/salir): ").strip().lower()
        
        if cmd == "ver":
            mostrar_libros_disponibles()
        
        elif cmd == "rango":
            filtrar_por_rango_precios()
        
        elif cmd == "comprar":
            titulo = input("Título del libro: ").strip()
            try:
                cantidad = int(input("Cantidad: ").strip())
            except ValueError:
                print("⚠️ Ingresa un número entero para la cantidad.")
                continue
            comprar_libros(titulo, cantidad)
        
        elif cmd == "salir":
            # Al salir, mostrar resumen (factura)
            print("\n===== FACTURA - RESUMEN DE COMPRA =====")
            if len(carrito) == 0:
                print("No se realizaron compras.")
            else:
                for i, (tit, cant, total_item, ahorro_item) in enumerate(carrito, start=1):
                    print(f"{i}. {tit} x{cant} | Total ítem: ${total_item:,.0f} | Ahorro: ${ahorro_item:,.0f}")
                print("---------------------------------------")
                print(f"Total de libros comprados: {total_libros_comprados}")
                print(f"Monto total pagado: ${total_pagado:,.0f}")
                print(f"Ahorro total por descuentos: ${total_ahorro:,.0f}")
            print("Gracias por tu compra en Libros & Bytes. ¡Vuelve pronto!")
            break
        
        else:
            print("Comando no reconocido. Usa: ver / rango / comprar / salir")

# Para iniciar la simulación interactiva, ejecuta:
# ejecutar_carrito()



---

## ¿Cómo usar este notebook?
1. Ejecuta la celda de **inventario** y las funciones (de arriba hacia abajo).
2. Opcional: ejecuta `mostrar_libros_disponibles()` o `filtrar_por_rango_precios()` para explorar.
3. Ejecuta `ejecutar_carrito()` para iniciar la simulación interactiva por consola (en la salida de la celda).

> **Nota:** Puedes editar el inventario o los descuentos directamente en la primera celda de código.
